In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

df = pd.read_csv(os.path.join('..', 'data', 'processed', 'master_clean.csv'),
                 parse_dates=['Start Date', 'End Date', 'Peak FL Date'])

print(df.shape)

(4548, 128)


In [2]:
new_cols = pd.DataFrame({
    'month'      : df['Start Date'].dt.month,
    'year'       : df['Start Date'].dt.year,
    'is_monsoon' : df['Start Date'].dt.month.between(6, 9).astype(int),
    'label'      : (df['Flood Type'] == 'Severe Flood').astype(int),
})

df = pd.concat([df, new_cols], axis=1)
df = df.copy()

print(df['label'].value_counts())
print(f"Severe Flood rate: {df['label'].mean()*100:.1f}%")

label
0    2919
1    1629
Name: count, dtype: int64
Severe Flood rate: 35.8%


In [3]:
new_precip = pd.DataFrame({
    'rain_day1'      : df['T1d'],
    'rain_day2'      : df['T2d'] - df['T1d'],
    'rain_day3'      : df['T3d'] - df['T2d'],
    'rain_3d_total'  : df['T3d'],
    'rain_7d_total'  : df['T7d'],
    'rain_10d_total' : df['T10d'],
    'rain_frontload' : df['T3d'] / (df['T10d'] + 1e-6),

    # cap at 99th percentile to kill extreme outliers from near-zero T3d
    'rain_accel'     : ((df['T10d'] - df['T7d']) / (df['T3d'] + 1e-6)).clip(
                        upper=((df['T10d'] - df['T7d']) / (df['T3d'] + 1e-6)).quantile(0.99))
})

df = pd.concat([df, new_precip], axis=1)
df = df.copy()

print(df[['rain_frontload', 'rain_accel']].describe().round(2))

       rain_frontload  rain_accel
count         4548.00     4548.00
mean             0.43        1.12
std              0.22        2.50
min              0.00        0.00
25%              0.27        0.20
50%              0.41        0.50
75%              0.57        1.06
max              1.00       20.06


In [4]:
# how flood-prone is this gauge historically?
# count of past events per gauge = proxy for chronic flood risk
flood_counts = df.groupby('GaugeID')['EventID'].transform('count')

df = pd.concat([df, pd.DataFrame({'gauge_flood_count': flood_counts})], axis=1)
df = df.copy()

print(df['gauge_flood_count'].describe().round(1))

count    4548.0
mean      131.8
std       120.8
min         1.0
25%        34.0
50%        83.0
75%       219.0
max       419.0
Name: gauge_flood_count, dtype: float64


In [5]:
out_path = os.path.join('..', 'data', 'processed', 'features.csv')
df.to_csv(out_path, index=False)
print(f"Saved: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

Saved: (4548, 141)
Label distribution:
label
0    2919
1    1629
Name: count, dtype: int64
